In [ ]:
!date

### high memory issues
- https://github.com/aertslab/scenicplus/issues/529


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

import scanpy as sc
import anndata

import pycisTopic
from pycisTopic.diff_features import (
    impute_accessibility,
    normalize_scores,
    find_highly_variable_features,
    find_diff_features
)
import os
import pyranges as pr
from pycisTopic.gene_activity import get_gene_activity
from scipy.sparse import csr_matrix
from pycisTopic.utils import region_names_to_coordinates
from pycisTopic.loom import export_region_accessibility_to_loom, export_gene_activity_to_loom

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
samples = list(np.loadtxt(f'{projdir}/txt/samples.txt', dtype=str))

In [ ]:
indir = f'{projdir}/atac/pycistopic/final_model'
n = 75
with open(f'{indir}/merged.intersect.{n}.preprocess.pkl', 'rb') as fin:
    cistopic_obj = pickle.load(fin)

In [ ]:
cistopic_obj

In [ ]:
imputed_acc_obj = impute_accessibility(
    cistopic_obj,
    selected_cells=None,
    selected_regions=None,
    scale_factor=10**6
)

In [ ]:
dir(imputed_acc_obj)

In [ ]:
# imputed_acc_obj.mtx = csr_matrix(imputed_acc_obj.mtx)

In [ ]:
help(normalize_scores)

In [ ]:
### not sure how to handle this
normalized_imputed_acc_obj = normalize_scores(imputed_acc_obj, scale_factor=10**4)

### can I subset peaks...?

In [ ]:
variable_regions = find_highly_variable_features(
    normalized_imputed_acc_obj,
    min_disp = 0.05,
    min_mean = 0.0125,
    max_mean = 3,
    max_disp = np.inf,
    n_bins=20,
    n_top_features=None,
    plot=True
)

In [ ]:
len(variable_regions)

In [ ]:
tmp_dir = '/u/project/cluo_scratch/terencew/tmp/scenic/'
markers_dict= find_diff_features(
    cistopic_obj,
    imputed_acc_obj,
    variable='clusters_v6',
    var_features=variable_regions,
    contrasts=None,
    adjpval_thr=0.05,
    log2fc_thr=np.log2(1.5),
    n_cpu=5,
    _temp_dir=tmp_dir,
    split_pattern = '-'
)

In [ ]:
markers_dict['Fail1_1'].shape

In [ ]:
imputed_acc_obj

### I think this is the minimum set of inputs for

In [ ]:
outdir = f'{projdir}/atac/pycistopic/final_model'
os.makedirs(os.path.join(outdir, "region_sets"), exist_ok = True)

In [ ]:
for cell_type in markers_dict:
    region_names_to_coordinates(
        markers_dict[cell_type].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(outdir, "region_sets", "DARs_cell_type", f"{cell_type}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [ ]:
cluster_markers = {'clusters_v6': markers_dict}

In [ ]:
os.makedirs(os.path.join(outdir, "loom"), exist_ok=True)

In [ ]:
# export_region_accessibility_to_loom(
#     accessibility_matrix = imputed_acc_obj,
#     cistopic_obj = cistopic_obj,
#     binarized_topic_region = region_bin_topics_otsu,
#     binarized_cell_topic = binarized_cell_topic,
#     selected_cells = cistopic_obj.projections['cell']['UMAP'].index.tolist(),
#     out_fname = os.path.join(outdir, "loom", "intersect_rna.loom"),
#     cluster_annotation = [''],
#     cluster_markers = cluster_markers,
#     tree_structure = ('10x_multiome_brain', 'pycisTopic', 'noDBL_all'),
#     title = 'Tutorial - Region accessibility all',
#     nomenclature = "hg38",
#     split_pattern = '-'
# )

In [ ]:
outdir = f'{projdir}/atac/pycistopic/final_model'
pickle.dump(imputed_acc_obj, open(f'{outdir}/merged.intersect.{n}.impute.pkl', 'wb')) 

In [ ]:
outdir = f'{projdir}/atac/pycistopic/final_model'
pickle.dump(imputed_acc_obj, open(f'{outdir}/merged.intersect.{n}.impute.pkl', 'wb')) 

In [ ]:
outdir = f'{projdir}/atac/pycistopic/final_model'
pickle.dump(normalized_imputed_acc_obj, open(f'{outdir}/merged.intersect.{n}.impute.normalize.pkl', 'wb')) 